# Dummy Audio Classification — Local Sanity Check

이 프로젝트의 `dummy_datasets/`를 로컬에서 직접 읽습니다.
첫 설정 cell의 `TASK`만 바꾼 뒤 위에서 아래로 실행하세요. accuracy는 의미가 없습니다.


In [ ]:
from dataclasses import dataclass
from pathlib import Path
import copy
import random

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import soundfile as sf
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.model_selection import StratifiedGroupKFold
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from torchvision.models import ResNet18_Weights, resnet18

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
TASK = "keyword_spotting"  # celebrity_voice | keyword_spotting | dog_breed

def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "dummy_datasets").is_dir():
            return candidate
    fixed = Path("/mnt/ssd4tb/lecture/haegon/audio_classification_projects")
    if (fixed / "dummy_datasets").is_dir():
        return fixed
    raise FileNotFoundError("dummy_datasets 폴더가 있는 프로젝트 루트에서 notebook을 실행하세요.")

PROJECT_DIR = find_project_root(Path.cwd().resolve())
PROJECT_ROOT = PROJECT_DIR / "dummy_datasets" / TASK
print("task:", TASK)
print("project root:", PROJECT_ROOT)
print("device:", DEVICE)



## 1. 과제 설정

`TASK`만 바꾸면 권장 설정이 달라집니다. 모든 과제의 입력은 16 kHz, mono, 3초입니다.



In [ ]:
TASK_PRESETS = {
    "celebrity_voice": {"num_classes": 30, "seconds": 3.0, "group_meaning": "source_video_id"},
    "keyword_spotting": {"num_classes": 11, "seconds": 1.5, "group_meaning": "speaker_id or room_session_id"},
    "dog_breed": {"num_classes": 10, "seconds": 3.0, "group_meaning": "dog_id or source_video_id"},
}

@dataclass
class Config:
    task: str = TASK
    data_root: str = str(PROJECT_ROOT)
    metadata_csv: str = str(PROJECT_ROOT / "metadata.csv")
    sample_rate: int = 16_000
    n_fft: int = 512
    win_length: int = 512
    hop_length: int = 160
    n_mels: int = 128
    batch_size: int = 32
    epochs: int = 1
    learning_rate: float = 3e-4
    num_workers: int = 0
    use_pretrained: bool = False
    use_augmentation: bool = True

CFG = Config()
PRESET = TASK_PRESETS[CFG.task]
CLIP_SAMPLES = int(CFG.sample_rate * PRESET["seconds"])
print(CFG)
print("group column means:", PRESET["group_meaning"])



## metadata.csv 초안 자동 생성

`audio/class_name/*.wav` 구조라면 아래 함수가 `path`와 `label`을 먼저 채웁니다.
`group`은 임시로 파일명을 넣으므로 실제 speaker/source video/dog ID로 반드시 고쳐야 합니다.
기존 `metadata.csv`를 덮어쓰지 않도록 저장 줄은 기본적으로 주석 처리되어 있습니다.



In [ ]:
def create_metadata_draft(project_root: Path) -> pd.DataFrame:
    rows = []
    for wav_path in sorted((project_root / "audio").glob("*/*.wav")):
        rows.append({
            "path": wav_path.relative_to(project_root).as_posix(),
            "label": wav_path.parent.name,
            "group": wav_path.stem,
            "split": "",
            "source_url": "",
            "license": "",
            "annotator": "",
            "notes": "group을 실제 source ID로 수정",
        })
    return pd.DataFrame(rows)

metadata_draft = create_metadata_draft(Path(CFG.data_root))
print(metadata_draft.head())
print("draft rows:", len(metadata_draft))
# metadata_draft.to_csv(Path(CFG.data_root) / "metadata_draft.csv", index=False, encoding="utf-8-sig")



## 2. metadata.csv 검사와 group-aware split

필수 열은 `path,label,group`입니다. 같은 `group`은 절대로 여러 split에 걸치지 않습니다.



In [ ]:
def make_group_splits(df: pd.DataFrame) -> pd.DataFrame:
    """Create 60/20/20-ish splits without sharing a group across splits."""
    df = df.copy()
    required = {"path", "label", "group"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"metadata.csv에 필요한 열이 없습니다: {sorted(missing)}")
    if df[["path", "label", "group"]].isna().any().any():
        raise ValueError("path, label, group에는 빈 값이 없어야 합니다.")
    if "split" in df and df["split"].notna().any():
        if not df["split"].notna().all() or set(df["split"].unique()) != {"train", "val", "test"}:
            raise ValueError("split을 직접 지정할 때는 모든 행을 train/val/test 중 하나로 채우세요.")
        return df

    per_label_groups = df.groupby("label")["group"].nunique()
    if per_label_groups.min() < 5:
        raise ValueError(
            "자동 split에는 label마다 서로 다른 group이 최소 5개 필요합니다. "
            "직접 split을 지정하거나 데이터를 더 모으세요.\n" + str(per_label_groups)
        )
    outer = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
    train_val_idx, test_idx = next(outer.split(df, df.label, groups=df.group))
    train_val = df.iloc[train_val_idx]
    inner = StratifiedGroupKFold(n_splits=4, shuffle=True, random_state=SEED + 1)
    train_rel, val_rel = next(inner.split(train_val, train_val.label, groups=train_val.group))
    df["split"] = "test"
    df.loc[train_val.iloc[train_rel].index, "split"] = "train"
    df.loc[train_val.iloc[val_rel].index, "split"] = "val"
    return df

def validate_splits(df: pd.DataFrame) -> None:
    for a, b in [("train", "val"), ("train", "test"), ("val", "test")]:
        overlap = set(df.loc[df.split == a, "group"]) & set(df.loc[df.split == b, "group"])
        assert not overlap, f"group leakage: {a}/{b}: {sorted(overlap)[:5]}"
    table = pd.crosstab(df.label, df.split)
    print(table)
    print("clips:", df.split.value_counts().to_dict())

metadata = pd.read_csv(CFG.metadata_csv)
metadata = make_group_splits(metadata)
validate_splits(metadata)
LABELS = sorted(metadata.label.unique())
label_to_idx = {label: i for i, label in enumerate(LABELS)}
assert len(LABELS) == PRESET["num_classes"], (len(LABELS), PRESET["num_classes"])



## 3. WAV → 1-channel log-mel spectrogram

ResNet-18은 고정된 224 × 224 입력을 요구하지 않습니다.
clip 길이를 먼저 고정했으므로 log-mel의 원래 크기 `[1, n_mels, time]`를 그대로 사용합니다.



In [ ]:
class AudioDataset(Dataset):
    def __init__(self, frame: pd.DataFrame, train: bool):
        self.frame = frame.reset_index(drop=True)
        self.train = train
        self.mel = torchaudio.transforms.MelSpectrogram(
            sample_rate=CFG.sample_rate,
            n_fft=CFG.n_fft,
            win_length=CFG.win_length,
            hop_length=CFG.hop_length,
            n_mels=CFG.n_mels,
            f_min=40,
            f_max=CFG.sample_rate // 2,
            power=2.0,
        )
        self.to_db = torchaudio.transforms.AmplitudeToDB(stype="power", top_db=80)

    def __len__(self):
        return len(self.frame)

    def _load_clip(self, path: Path) -> torch.Tensor:
        audio, sr = sf.read(path, dtype="float32", always_2d=True)
        wav = torch.from_numpy(audio.T).mean(dim=0, keepdim=True)
        if sr != CFG.sample_rate:
            wav = torchaudio.functional.resample(wav, sr, CFG.sample_rate)
        if wav.shape[-1] < CLIP_SAMPLES:
            wav = F.pad(wav, (0, CLIP_SAMPLES - wav.shape[-1]))
        elif wav.shape[-1] > CLIP_SAMPLES:
            if self.train:
                start = random.randint(0, wav.shape[-1] - CLIP_SAMPLES)
            else:
                start = (wav.shape[-1] - CLIP_SAMPLES) // 2
            wav = wav[..., start:start + CLIP_SAMPLES]
        return wav

    def _augment_wave(self, wav: torch.Tensor) -> torch.Tensor:
        if not (self.train and CFG.use_augmentation):
            return wav
        wav = wav * random.uniform(0.7, 1.3)
        wav = torch.roll(wav, random.randint(-1600, 1600), dims=-1)
        if random.random() < 0.5:
            wav = wav + torch.randn_like(wav) * random.uniform(0.001, 0.015)
        return wav.clamp(-1, 1)

    def __getitem__(self, index):
        row = self.frame.iloc[index]
        wav = self._augment_wave(self._load_clip(Path(CFG.data_root) / row.path))
        spec = self.to_db(self.mel(wav))
        spec = ((spec + 80.0) / 80.0).clamp(0, 1)
        if self.train and CFG.use_augmentation:
            spec = torchaudio.transforms.FrequencyMasking(12)(spec)
            spec = torchaudio.transforms.TimeMasking(18)(spec)
        spec = (spec - 0.5) / 0.25
        return spec, label_to_idx[row.label], str(row.path)

datasets = {
    split: AudioDataset(metadata[metadata.split == split], train=(split == "train"))
    for split in ["train", "val", "test"]
}

train_labels = metadata.loc[metadata.split == "train", "label"].map(label_to_idx).to_numpy()
counts = np.bincount(train_labels, minlength=len(LABELS))
sample_weights = 1.0 / np.maximum(counts[train_labels], 1)
sampler = WeightedRandomSampler(sample_weights, len(sample_weights), replacement=True)
loaders = {
    "train": DataLoader(datasets["train"], batch_size=CFG.batch_size, sampler=sampler,
                        num_workers=CFG.num_workers, pin_memory=True),
    "val": DataLoader(datasets["val"], batch_size=CFG.batch_size, shuffle=False,
                      num_workers=CFG.num_workers, pin_memory=True),
    "test": DataLoader(datasets["test"], batch_size=CFG.batch_size, shuffle=False,
                       num_workers=CFG.num_workers, pin_memory=True),
}



## 4. ImageNet ResNet-18의 첫 층을 3채널 → 1채널로 교체



In [ ]:
def build_model(num_classes: int) -> nn.Module:
    weights = ResNet18_Weights.DEFAULT if CFG.use_pretrained else None
    model = resnet18(weights=weights)
    old_weight = model.conv1.weight.detach().mean(dim=1, keepdim=True)
    model.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
    if CFG.use_pretrained:
        model.conv1.weight.data.copy_(old_weight)
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model

model = build_model(len(LABELS)).to(DEVICE)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=CFG.learning_rate, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CFG.epochs)
print(model.conv1)
print(model.fc)



## 5. 학습: validation macro-F1이 가장 높은 checkpoint 저장



In [ ]:
@torch.no_grad()
def evaluate(loader):
    model.eval()
    ys, preds, paths = [], [], []
    total_loss = 0.0
    for x, y, batch_paths in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        logits = model(x)
        total_loss += criterion(logits, y).item() * len(y)
        ys.extend(y.cpu().tolist())
        preds.extend(logits.argmax(1).cpu().tolist())
        paths.extend(batch_paths)
    return {
        "loss": total_loss / len(loader.dataset),
        "accuracy": accuracy_score(ys, preds),
        "macro_f1": f1_score(ys, preds, average="macro", zero_division=0),
        "y": ys,
        "pred": preds,
        "paths": paths,
    }

best_state, best_f1 = None, -1.0
history = []
for epoch in range(1, CFG.epochs + 1):
    model.train()
    running = 0.0
    for x, y, _ in loaders["train"]:
        x, y = x.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        loss = criterion(model(x), y)
        loss.backward()
        optimizer.step()
        running += loss.item() * len(y)
    val = evaluate(loaders["val"])
    train_loss = running / len(datasets["train"])
    history.append({"epoch": epoch, "train_loss": train_loss, **{f"val_{k}": v for k, v in val.items() if k in ["loss", "accuracy", "macro_f1"]}})
    print(f"{epoch:02d} train_loss={train_loss:.4f} val_acc={val['accuracy']:.3f} val_macro_f1={val['macro_f1']:.3f}")
    if val["macro_f1"] > best_f1:
        best_f1 = val["macro_f1"]
        best_state = copy.deepcopy(model.state_dict())
    scheduler.step()

model.load_state_dict(best_state)
torch.save({"model": model.state_dict(), "labels": LABELS, "config": CFG.__dict__}, "best_resnet18_audio.pt")



## 6. test 평가는 마지막에 한 번만



In [ ]:
test = evaluate(loaders["test"])
print({k: test[k] for k in ["loss", "accuracy", "macro_f1"]})
print(classification_report(test["y"], test["pred"], target_names=LABELS, zero_division=0))

cm = confusion_matrix(test["y"], test["pred"], normalize="true")
plt.figure(figsize=(10, 8))
sns.heatmap(cm, cmap="Blues", vmin=0, vmax=1, xticklabels=LABELS, yticklabels=LABELS)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Normalized confusion matrix")
plt.tight_layout()
plt.show()



## 7. 실패 사례를 직접 듣고 원인을 기록

confusion matrix만 제출하지 말고, 오분류 파일을 최소 20개 들어 본 뒤
`잡음 / 겹침 / 짧음 / 라벨 오류 / 출처 편향 / 실제로 애매함` 중 원인을 annotation 하세요.



In [ ]:
errors = pd.DataFrame({"path": test["paths"], "true_idx": test["y"], "pred_idx": test["pred"]})
errors = errors[errors.true_idx != errors.pred_idx].copy()
errors["true_label"] = errors.true_idx.map(dict(enumerate(LABELS)))
errors["pred_label"] = errors.pred_idx.map(dict(enumerate(LABELS)))
errors["error_reason"] = ""
errors.to_csv("error_analysis.csv", index=False)
errors.head(20)
